In [ ]:
%pip install -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-groq \
    langchain-text-splitters \
    faiss-cpu \
    python-dotenv \
    pypdf \
    requests \
    jupyter \
    ipykernel


In [71]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import JinaEmbeddings

# Package -> langchain_community
# class -> document_loaders
# method -> TextLoader

load_dotenv()

print("All imports successful!")

All imports successful!


In [72]:
# Groq serves the chat model below. Keep the XAI_API_KEY fallback for existing local setups.
groq_api_key = os.getenv("GROQ_API_KEY") or os.getenv("XAI_API_KEY")
jina_api_key = os.getenv("JINA_API_KEY")

if not groq_api_key:
    raise ValueError("Set GROQ_API_KEY in your environment before running the chat model.")
if not jina_api_key:
    raise ValueError("Set JINA_API_KEY in your environment before creating embeddings.")

print("Required API keys are configured.")


Required API keys are configured.


In [73]:
# Resolve data/ correctly whether Jupyter starts in Backend/ or the workspace root.
from pathlib import Path

notebook_dir = Path.cwd()
if not (notebook_dir / "data" / "my_resume.txt").is_file():
    notebook_dir = notebook_dir / "Backend"

data_file_path = notebook_dir / "data" / "my_resume.txt"
if not data_file_path.is_file():
    raise FileNotFoundError(f"Resume data file was not found at {data_file_path}")


In [74]:
#Data Ingestion
text_loader = TextLoader(data_file_path, encoding='utf-8')

#Langchain processing everything in form of documents (Page Content and Metadata)
documents = text_loader.load()

print(f"Number of documents loaded: {len(documents)}")

Number of documents loaded: 1


In [75]:
print(documents[0].metadata)
print(f"size of our data: {len(documents[0].page_content)}")

{'source': '/Users/amitprasad/Amit Portfolio/Backend/data/my_resume.txt'}
size of our data: 7766


In [76]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split the data into chunks for better processing and retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# Split your loaded LangChain documents
chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
# print(chunks[0].page_content)

Total chunks created: 11


In [77]:
# Retry transient Jina API disconnects and bound each request's wait time.
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from langchain_community.embeddings import JinaEmbeddings

class TimeoutSession(requests.Session):
    def request(self, method, url, **kwargs):
        kwargs.setdefault("timeout", (10, 60))
        return super().request(method, url, **kwargs)

retry_policy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"POST"}),
)
embedding_session = TimeoutSession()
embedding_session.mount("https://", HTTPAdapter(max_retries=retry_policy))

embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en",
    jina_api_key=jina_api_key,
    session=embedding_session,
)

print(f"Embedding model is ready: {embeddings_model.model_name}")


Embedding model is ready: jina-embeddings-v2-base-en


In [78]:
#Store data in vector database using FAISS (Fast AI Semantic Search)
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)

print("Chunks were stored", vector_store.index.ntotal, "in the vector database.")

Chunks were stored 11 in the vector database.


In [79]:
test_query = "What is your name?"

top_matches = vector_store.similarity_search(test_query, k=3)
print("Top matches found:")
for i, match in enumerate(top_matches):
    print(f"---- Match {i+1} ------")
    print(match.page_content)   



Top matches found:
---- Match 1 ------
National Cadet Corps (NCC):

NCC 'C' Certificate with ALPHA grading, 2019.
NCC 'B' Certificate with BETA grading, 2018.

HOBBIES AND INTERESTS

Table Tennis
Basketball
Running

AREAS OF EXPERTISE

Retrieval-Augmented Generation (RAG) Systems
Large Language Model (LLM) Applications
Semantic Search and Vector Retrieval
Document Question Answering
PDF Parsing and Document Chunking
Embedding Models and Similarity Search
LangChain and LangGraph
Agentic AI and Multi-Agent Workflows
Prompt Engineering and Context Management
Structured Information Extraction
Pydantic Schema Validation
Azure OpenAI and Azure AI Search
FastAPI and REST API Development
Full-Stack Application Development
React.js and Python
Docker and CI/CD Automation
Open-Source LLM Integration
Machine Learning and Predictive Modeling
Enterprise AI Assistants
Workflow Automation
---- Match 2 ------
Client: Hewlett Packard Enterprise (HPE)

Project: HPE Aruba SD-WAN Orchestrator

Developed ne

In [80]:
from langchain.tools import tool

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

@tool
def search_my_resume(query: str) -> str:
    """Search my resume for relevant information about skills, experience, education, and projects."""

    retrieved_docs = retriever.invoke(query)

    matching_chunks = [
        doc.page_content for doc in retrieved_docs
    ]

    if not matching_chunks:
        return "No relevant information found in the resume."

    return "\n\n".join(matching_chunks)

In [81]:
# LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5,
    api_key=groq_api_key,
)

llm.model_name


'openai/gpt-oss-120b'

In [82]:
tr = llm.invoke("What is your name?")

In [83]:
tr.content

'I’m ChatGPT, an AI language model created by OpenAI. How can I help you today?'

In [84]:
# AI Agent
# LLM - Brain
# Tool - Super Power
# Memory - no memory

In [85]:
from langchain.agents import create_agent

In [86]:
from langchain.agents import create_agent

my_assistant = create_agent(
    model=llm,
    tools=[search_my_resume],
    system_prompt="You are a helpful assistant that can answer questions about my resume."
)

print("Your AI assistant is ready to use!")

Your AI assistant is ready to use!


In [87]:
def ask_my_resume(query: str) -> str:
    """Ask questions about my resume and get relevant answers."""

    response = my_assistant.invoke(
        {"messages": [{"role": "user", "content": query}]}
    )
    answer = response["messages"][-1].content
    return answer


In [88]:
response = my_assistant.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Currently where he is working and what is his designation?"
            }
        ]
    }
)

print(response["messages"][-1].content)

In [89]:
response["messages"][-2].content

"National Cadet Corps (NCC):\n\nNCC 'C' Certificate with ALPHA grading, 2019.\nNCC 'B' Certificate with BETA grading, 2018.\n\nHOBBIES AND INTERESTS\n\nTable Tennis\nBasketball\nRunning\n\nAREAS OF EXPERTISE\n\nRetrieval-Augmented Generation (RAG) Systems\nLarge Language Model (LLM) Applications\nSemantic Search and Vector Retrieval\nDocument Question Answering\nPDF Parsing and Document Chunking\nEmbedding Models and Similarity Search\nLangChain and LangGraph\nAgentic AI and Multi-Agent Workflows\nPrompt Engineering and Context Management\nStructured Information Extraction\nPydantic Schema Validation\nAzure OpenAI and Azure AI Search\nFastAPI and REST API Development\nFull-Stack Application Development\nReact.js and Python\nDocker and CI/CD Automation\nOpen-Source LLM Integration\nMachine Learning and Predictive Modeling\nEnterprise AI Assistants\nWorkflow Automation\n\nClient: Hewlett Packard Enterprise (HPE)\n\nProject: HPE Aruba SD-WAN Orchestrator\n\nDeveloped new features, includi